In [1]:
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)


In [10]:
from typing import Annotated, TypedDict
from langchain_core.messages import AnyMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# -- State ---------------------------------------------------------------------
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

In [11]:
# -- Tools ---------------------------------------------------------------------
@tool
def add(a: int, b: int) -> int:
    """Add two integers and return the result."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the result."""
    return a * b


tools = [add, multiply]
llm_with_tools = llm.bind_tools(tools)

# -- Node: Call the model ------------------------------------------------------
def generate(state: State) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

In [12]:
# -- Build Graph ---------------------------------------------------------------
builder = StateGraph(State)

builder.add_node("chatbot", generate)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition)
builder.add_edge("tools", "chatbot")

graph = builder.compile()
print("Tools chatbot graph compiled successfully.")

Tools chatbot graph compiled successfully.


In [15]:
# -- Prediction Step -----------------------------------------------------------
input_state = {
    "messages": [
        HumanMessage(content="What is 12 plus 8? Then multiply the result by 3.")
    ]
}

prediction = graph.invoke(input_state)
print(prediction)

# for message in prediction["messages"]:
    # message.pretty_print()

{'messages': [HumanMessage(content='What is 12 plus 8? Then multiply the result by 3.', additional_kwargs={}, response_metadata={}, id='9965dfd4-3fe8-4629-98c5-2993f375e58a'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user is asking to first add 12 and 8, then multiply the result by 3. \n\nSo, the first step is to calculate 12 plus 8. The add function can handle that. The parameters needed are a and b, both integers. Here, a is 12 and b is 8. \n\nOnce I get the sum, which should be 20, the next step is to multiply that by 3. The multiply function requires two integers as well. The sum from the first step will be the a parameter, and 3 will be the b parameter.\n\nI need to make sure to call the add function first, then use its result in the multiply function. Since the tools provided include both add and multiply, I can perform these steps sequentially. \n\nWait, but how do I handle the result of the add function before using it in multiply? Si

In [16]:
input_state

{'messages': [HumanMessage(content='What is 12 plus 8? Then multiply the result by 3.', additional_kwargs={}, response_metadata={}, id='9965dfd4-3fe8-4629-98c5-2993f375e58a')]}